# 0413 · RandomForest (ensamble)

**Qué hace.** Entrena y valida solo RandomForest, en las dos tareas:

| Tarea | Qué predice | Métrica para elegir |
| --- | --- | --- |
| `clf` | Probabilidad de incumplir el SLA | AP (mayor es mejor) |
| `reg` | Horas de última milla | MAE en horas (menor es mejor) |

**Random Forest** promedia muchos árboles de decisión entrenados con muestras distintas de guías y de variables. Es el ensamble más simple de los tres y el más lento de predecir con muchas filas.

**Pasos.** 1) carga lo que dejó 0411; 2) búsqueda aleatoria de hiperparámetros con validación de origen móvil; 3) validación con la mejor configuración en los tres bloques (abril, mayo, junio); 4) vista rápida de sus métricas; 5) guarda predicciones, importancias y parámetros para 0416.

**No depende de los otros cuadernos de modelos.** Solo necesita que 0411 se haya corrido. La prueba final no se toca aquí.

**Tiempo.** La búsqueda hace `N_CONFIGS_BUSQUEDA × 3 bloques × 2 tareas` ajustes sobre una submuestra de `MAX_FILAS_BUSQUEDA` filas. Para una primera corrida rápida, bajar `N_CONFIGS_BUSQUEDA` a 3.

In [0]:
# Si faltan librerías:
# !pip install duckdb lightgbm xgboost scikit-learn joblib

In [0]:
%pip install "duckdb<2" lightgbm xgboost shap
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 103.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 48.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 121.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.3/547.3 kB 18.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 126.8 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [shap]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 0. Configuración

In [0]:
import os, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)

# MODULOS/ debe estar junto al cuaderno, como en 0100 (MODULOS/conexion_sql.py)
from MODULOS.modelado_temporal import (cargar_preparacion, pliegues_origen_movil, partes_del_bloque, marco_predicciones,
                                       construir_modelo, predecir, muestrear_configs, puntaje_config, mejores, params_de,
                                       importancia_pliegue, tabla_metricas)

In [0]:
MODELO = "RandomForest"
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_10/V3/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"

N_CONFIGS_BUSQUEDA = 20        # configuraciones aleatorias por tarea
MAX_FILAS_BUSQUEDA = 300_000   # submuestra de entrenamiento en la búsqueda (no en la validación)
EJECUTAR_BUSQUEDA  = True      # False: usa los parámetros que ya guardó este cuaderno
N_PERMUTACION      = 10_000    # filas de validación para la importancia por permutación en cada bloque

In [0]:
# Hiperparámetros de base (si no hay búsqueda) y espacio de búsqueda registrado (realimentación P4)
PARAMS_BASE = dict(n_estimators=200, min_samples_leaf=20, max_features="sqrt", max_samples=0.2)
ESPACIO = dict(n_estimators=[100, 200, 300], max_depth=[None, 12, 20], min_samples_leaf=[1, 5, 20, 50],
               max_features=["sqrt", 0.3, 0.5], max_samples=[0.1, 0.2, 0.3])

## 1. Cargar la preparación de 0411

Base de modelado, historia sin futuro de cada bloque y configuración común (periodos, semilla, variables). Los pliegues se vuelven a armar con la misma función del asesor.

In [0]:
CFG, df, HIST = cargar_preparacion(CARPETA_MODELADO)
SEMILLA, CATEGORICAS, VERSIONES, TAREAS = CFG["SEMILLA"], CFG["CATEGORICAS"], CFG["VERSIONES"], CFG["TAREAS"]
PLIEGUES = {t: list(pliegues_origen_movil(df, CFG["INICIOS_VALIDACION"], CFG["FIN_DESARROLLO"], t_etiqueta=te))
            for t, _, te in TAREAS}
print(f"{len(df):,} guías · versiones: {list(VERSIONES)} · preparado el {CFG['fecha_preparacion']}")

1,588,872 guías · versiones: ['internas', 'internas+clima'] · preparado el 2026-10-24T18:56:48.281707


## 2. Búsqueda aleatoria de hiperparámetros (solo en validación)

Se prueban `N_CONFIGS_BUSQUEDA` configuraciones de `ESPACIO`. Cada una se entrena en cada bloque con una submuestra de hasta `MAX_FILAS_BUSQUEDA` filas y se evalúa en el bloque completo. Se elige la de mejor promedio entre bloques. Si la base trae la versión «internas+clima_reajustada», se busca también para ella.

In [0]:
def buscar(version, numericas):
    """Prueba todas las configuraciones de una versión de variables; una fila por configuración."""
    filas = []
    for tarea, y_col, _ in TAREAS:
        for k, cfg in enumerate(muestrear_configs(ESPACIO, N_CONFIGS_BUSQUEDA, SEMILLA + len(MODELO))):
            if tarea == "clf":
                cfg = {a: b for a, b in cfg.items() if a != "objetivo_reg"}   # solo aplica a regresión
            vals = puntaje_config(tarea, MODELO, cfg, k, PLIEGUES[tarea], HIST, df, y_col,
                                  CATEGORICAS, numericas, MAX_FILAS_BUSQUEDA, SEMILLA)
            filas.append({"version": version, "tarea": tarea, "modelo": MODELO, "config": k,
                          "metrica": "AP" if tarea == "clf" else "MAE_h", "promedio": np.mean(vals),
                          "por_bloque": [round(v, 4) for v in vals], **{f"p_{a}": b for a, b in cfg.items()}})
            print(f"{version} {tarea} config {k}: {np.mean(vals):.4f}")
    return pd.DataFrame(filas)

In [0]:
RUTA_PARAMS = f"{CARPETA_MODELADO}parametros_{MODELO}.json"
t_ini = time.time()
if EJECUTAR_BUSQUEDA:
    versiones_busqueda = ["internas"] + [v for v in VERSIONES if v == "internas+clima_reajustada"]
    tabla_busqueda = pd.concat([buscar(v, VERSIONES[v]) for v in versiones_busqueda], ignore_index=True)
    tabla_busqueda.to_pickle(f"{CARPETA_MODELADO}busqueda_{MODELO}.pkl")       # exacta, para 0416
    tabla_busqueda.to_csv(f"{CARPETA_MODELADO}busqueda_{MODELO}.csv", index=False)
    MEJORES = mejores(tabla_busqueda, {MODELO: ESPACIO})
else:
    MEJORES = json.load(open(RUTA_PARAMS))["mejores"]
print(f"Búsqueda: {time.time() - t_ini:.0f} s")
display(pd.DataFrame(MEJORES).T)

internas clf config 0: 0.3463
internas clf config 1: 0.3565
internas clf config 2: 0.3516
internas clf config 3: 0.3540
internas clf config 4: 0.3538
internas clf config 5: 0.3254
internas clf config 6: 0.3545
internas clf config 7: 0.3345
internas clf config 8: 0.3470
internas clf config 9: 0.3293
internas clf config 10: 0.3559
internas clf config 11: 0.3550
internas clf config 12: 0.3437
internas clf config 13: 0.3433
internas clf config 14: 0.3492
internas clf config 15: 0.3413
internas clf config 16: 0.3518
internas clf config 17: 0.3509
internas clf config 18: 0.3512
internas clf config 19: 0.3494
internas reg config 0: 8.2714
internas reg config 1: 7.8749
internas reg config 2: 7.9275
internas reg config 3: 7.7255
internas reg config 4: 7.9253
internas reg config 5: 8.9987
internas reg config 6: 7.6885
internas reg config 7: 9.0918
internas reg config 8: 8.4168
internas reg config 9: 9.1056
internas reg config 10: 7.7351
internas reg config 11: 7.8405
internas reg config 12: 7.61

n_estimators,max_depth,min_samples_leaf,max_features,max_samples
100.0,20.0,20.0,0.5,0.1
100.0,null,50.0,0.3,0.1


## 3. Validación de origen móvil con la mejor configuración

En cada versión de variables, tarea y bloque: se entrena con todo lo anterior al bloque, se predice el bloque y se guardan las predicciones. Con las variables internas se mide además la importancia por permutación (insumo de 0420).

In [0]:
pred, imp = [], []
t_ini = time.time()
for version, numericas in VERSIONES.items():
    for tarea, y_col, _ in TAREAS:
        params = params_de(MEJORES, {MODELO: PARAMS_BASE}, version, tarea, MODELO)
        for ini, tr, va in PLIEGUES[tarea]:
            Dva, Xtr, ytr, Xva, yva = partes_del_bloque(df, HIST[ini], y_col, tr, va, CATEGORICAS + numericas)
            m = construir_modelo(tarea, MODELO, numericas, CATEGORICAS, params, SEMILLA).fit(Xtr, ytr)
            pred.append(marco_predicciones(version, tarea, ini, Dva, yva).assign(modelo=MODELO, pred=predecir(m, Xva, tarea)))
            if version == "internas":
                etiquetas = {"tarea": tarea, "modelo": MODELO, "bloque": ini.date()}
                imp.append(importancia_pliegue(m, Xva, yva, tarea, N_PERMUTACION, SEMILLA, etiquetas))
            print(f"{version} · {tarea} · {ini.date()} listo ({time.time() - t_ini:.0f} s)")
pred, imp = pd.concat(pred, ignore_index=True), pd.concat(imp, ignore_index=True)

internas · clf · 2026-04-01 listo (50 s)
internas · clf · 2026-05-01 listo (113 s)
internas · clf · 2026-06-01 listo (194 s)
internas · reg · 2026-04-01 listo (224 s)
internas · reg · 2026-05-01 listo (267 s)
internas · reg · 2026-06-01 listo (321 s)
internas+clima · clf · 2026-04-01 listo (373 s)
internas+clima · clf · 2026-05-01 listo (448 s)
internas+clima · clf · 2026-06-01 listo (542 s)
internas+clima · reg · 2026-04-01 listo (573 s)
internas+clima · reg · 2026-05-01 listo (617 s)
internas+clima · reg · 2026-06-01 listo (676 s)


## 4. Vista rápida de RandomForest

Solo para revisar este modelo. La comparación con los demás se hace en 0416.

In [0]:
display(tabla_metricas(pred, ["version", "tarea", "modelo"]).round(4))
display(tabla_metricas(pred, ["version", "tarea", "modelo", "bloque"]).round(4))

version,tarea,modelo,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,RandomForest,587957,0.0709,0.3508,0.7412,0.0553,0.2152,null,null,null,null,null,null
internas,reg,RandomForest,587951,null,null,null,null,null,7.4763,36.9465,9.1104,43.1287,93.7313,0.9655
internas+clima,clf,RandomForest,587957,0.0709,0.3468,0.7444,0.0555,0.215,null,null,null,null,null,null
internas+clima,reg,RandomForest,587951,null,null,null,null,null,7.6666,36.9116,9.745,42.8181,100.4958,0.9655


version,tarea,modelo,bloque,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,RandomForest,2026-04-01,196990,0.064,0.3632,0.7479,0.0495,0.1981,null,null,null,null,null,null
internas,clf,RandomForest,2026-05-01,205304,0.0742,0.3694,0.7577,0.0567,0.218,null,null,null,null,null,null
internas,clf,RandomForest,2026-06-01,185663,0.0746,0.3259,0.7261,0.0599,0.2303,null,null,null,null,null,null
internas,reg,RandomForest,2026-04-01,196989,null,null,null,null,null,6.9052,37.3239,8.2575,36.6139,99.8912,0.967
internas,reg,RandomForest,2026-05-01,205301,null,null,null,null,null,7.3118,34.2298,9.5948,42.1233,89.364,0.9646
internas,reg,RandomForest,2026-06-01,185661,null,null,null,null,null,8.2641,39.3589,9.9671,51.1238,92.0088,0.965
internas+clima,clf,RandomForest,2026-04-01,196990,0.064,0.3504,0.7432,0.0499,0.1987,null,null,null,null,null,null
internas+clima,clf,RandomForest,2026-05-01,205304,0.0742,0.3656,0.7578,0.057,0.2184,null,null,null,null,null,null
internas+clima,clf,RandomForest,2026-06-01,185663,0.0746,0.3271,0.7342,0.0597,0.2287,null,null,null,null,null,null
internas+clima,reg,RandomForest,2026-04-01,196989,null,null,null,null,null,7.0334,37.3067,8.5702,36.6037,104.3369,0.967


In [0]:
# Variables con más peso en validación (promedio de los tres bloques)
display(imp.groupby(["tarea", "variable"]).importancia.mean().sort_values(ascending=False).groupby(level=0).head(8).round(4))

tarea  variable            
reg    hist_h_zona_franja      0.7199
       categoria_zona          0.6937
       hist_inc_zona_franja    0.5238
       margen_sla_en_t0_h      0.3607
       n_hist_cliente          0.2755
       hist_h_cliente          0.2222
       ID_PRODUCTO             0.2101
       ID_TIPOGUIA             0.2006
clf    margen_sla_en_t0_h      0.1690
       dia_semana              0.0722
       hist_inc_zona_franja    0.0696
       hist_h_zona_franja      0.0357
       categoria_zona          0.0245
       ID_PRODUCTO             0.0212
       ID_TIPOGUIA             0.0104
       hist_inc_cliente        0.0084
Name: importancia, dtype: float64

## 5. Guardar para 0416

In [0]:
pred.to_parquet(f"{CARPETA_MODELADO}pred_val_{MODELO}.parquet", index=False)
imp.to_parquet(f"{CARPETA_MODELADO}importancia_{MODELO}.parquet", index=False)
with open(RUTA_PARAMS, "w") as f:
    json.dump({"modelo": MODELO, "espacio": ESPACIO, "params_base": PARAMS_BASE, "semilla": SEMILLA,
               "n_configs": N_CONFIGS_BUSQUEDA, "max_filas": MAX_FILAS_BUSQUEDA, "mejores": MEJORES}, f, indent=2, default=str)
print("Guardado en", CARPETA_MODELADO)

Guardado en /Volumes/proyectogrado/source/source/LOG_CONSOLIDADO/REVISION_2026_10/V3/MODELADO/
